In [1]:
import numpy as np
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents

embedder = Embedder()

2026-06-28 10:21:36.158189227 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Q1: First value v[0] = {v[0]:.2f}")

Q1: First value v[0] = -0.02


In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents.")

Loaded 72 documents.


In [4]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc_q2 = next(d for d in documents if d['filename'] == target_filename)

v_doc = embedder.encode(doc_q2['content'])
similarity = np.dot(v, v_doc)

print(f"Q2: Cosine similarity = {similarity:.2f}")

Q2: Cosine similarity = 0.36


In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks.")

contents = [chunk['content'] for chunk in chunks]


X = embedder.encode_batch(contents)
X = np.array(X)


scores = X.dot(v)

best_chunk_idx = np.argmax(scores)
best_filename = chunks[best_chunk_idx]['filename']

print(f"Q3: File with highest-scoring chunk = {best_filename}")

Created 295 chunks.
Q3: File with highest-scoring chunk = 02-vector-search/lessons/07-sqlitesearch-vector.md


In [6]:
from minsearch import Index, VectorSearch

In [12]:
vector_index = VectorSearch()
vector_index.fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_v = embedder.encode(q4_query)
q4_results = vector_index.search(q4_v, num_results=1)

print(f"Q4 First result filename: {q4_results[0]['filename']}")

Q4 First result filename: 04-evaluation/lessons/05-search-metrics.md


In [15]:
text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"

q5_v = embedder.encode(q5_query)
q5_vector_results = vector_index.search(q5_v, num_results=5)
q5_text_results = text_index.search(q5_query,{}, num_results=5)

vector_files = {res['filename'] for res in q5_vector_results}
text_files = {res['filename'] for res in q5_text_results}

in_vector_not_text = vector_files - text_files

print(f"Q5 Files in vector results but not in text results:")
for file in in_vector_not_text:
    print(f"- {file}")

Q5 Files in vector results but not in text results:
- 02-vector-search/lessons/08-pgvector.md


In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6_query = "How do I give the model access to tools?"

q6_v = embedder.encode(q6_query)
q6_vector_results = vector_index.search(q6_v, num_results=5)
q6_text_results = text_index.search(q6_query,{}, num_results=5)

q6_hybrid_results = rrf([q6_vector_results, q6_text_results])

print(f"Q6 First result filename after RRF: {q6_hybrid_results[0]['filename']}")

Q6 First result filename after RRF: 01-agentic-rag/lessons/13-function-calling.md
